# Module 4: MapReduce Model & PySpark RDD Pipelines

**Z3003 Data Mining: Scaling to Large Data (BS Core) · IIT Madras Zanzibar**

Distributed map-shuffle-reduce abstractions, OLAP SUM/COUNT averaging, and HTTP server log parsing.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/dm-scale/labs/DM04_MapReduce_PySpark_Foundations.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every method is self-contained and verified with assertions.

## 1. The MapReduce Model (`map -> shuffle -> reduce`)

MapReduce expresses distributed aggregations as pure key-value transformations. Distributive measures (`SUM`, `COUNT`) and algebraic measures (`AVG = SUM / COUNT`) combine safely across cluster partitions.

In [1]:
from collections import defaultdict
import re


def run_mapreduce(records, mapper, reducer):
    grouped = defaultdict(list)
    for rec in records:
        for k, v in mapper(rec):
            grouped[k].append(v)
    return {k: reducer(k, vals) for k, vals in grouped.items()}


# Zanzibar farm-season yield aggregation (SUM, COUNT -> AVG without averaging averages!)
farm_seasons = [
    ("Pemba North", "Clove", 2.8), ("Pemba North", "Clove", 3.2),
    ("Pemba South", "Clove", 2.4), ("Unguja North", "Rice", 4.1),
    ("Unguja North", "Rice", 3.9), ("Unguja South", "Cassava", 8.5),
]
agg = run_mapreduce(
    farm_seasons,
    mapper=lambda r: [(r[1], (r[2], 1))],
    reducer=lambda crop, vs: (sum(x for x, _ in vs), sum(c for _, c in vs)),
)
for crop, (s, c) in sorted(agg.items()):
    print(f"{crop:8s} -> count={c}, sum={s:.1f} t, mean={s/c:.2f} t")
assert abs(agg["Clove"][0] / agg["Clove"][1] - 2.8) < 1e-9


Cassava  -> count=1, sum=8.5 t, mean=8.50 t
Clove    -> count=3, sum=8.4 t, mean=2.80 t
Rice     -> count=2, sum=8.0 t, mean=4.00 t


## 2. PySpark RDD Pipeline (with Automatic Local Python Fallback)

In Google Colab (`pip install pyspark`), this cell runs directly on `SparkContext.parallelize`; in lightweight local environments without a JVM, it falls back cleanly to the identical MapReduce semantics.

In [2]:
sample_logs = [
    '199.72.81.55 - - [01/Jul/1995:00:00:01 -0400] "GET /history/apollo/ HTTP/1.0" 200 6245',
    '199.72.81.55 - - [01/Jul/1995:00:00:02 -0400] "GET /missing.gif HTTP/1.0" 404 -',
    'piweba3y.prodigy.com - - [01/Jul/1995:00:00:05 -0400] "GET /ksc.html HTTP/1.0" 200 7074',
    'piweba3y.prodigy.com - - [01/Jul/1995:00:00:09 -0400] "GET /images/NASA.gif HTTP/1.0" 200 1204',
]
LOG_RE = re.compile(r'(\S+) \S+ \S+ \[([^\]]+)\] "(\S+) (\S+) \S+" (\d+) (\S+)')


def parse_line(line):
    m = LOG_RE.match(line)
    if not m:
        return None
    host, ts, _, path, status, sz = m.groups()
    return (host, path, int(status), 0 if sz == "-" else int(sz))


try:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.master("local[*]").appName("Z3003-DM04").getOrCreate()
    sc = spark.sparkContext
    sc.setLogLevel("ERROR")
    rdd = sc.parallelize(sample_logs).map(parse_line).filter(lambda x: x is not None)
    status_counts = dict(rdd.map(lambda r: (r[2], 1)).reduceByKey(lambda a, b: a + b).collect())
    mode = f"PySpark {sc.version}"
except Exception:
    parsed = [p for p in (parse_line(l) for l in sample_logs) if p is not None]
    status_counts = run_mapreduce(parsed, lambda r: [(r[2], 1)], lambda k, vs: sum(vs))
    mode = "Pure-Python MapReduce fallback"

print(f"Executed via {mode}: status_counts = {status_counts}")
assert status_counts == {200: 3, 404: 1}


Executed via Pure-Python MapReduce fallback: status_counts = {200: 3, 404: 1}
